# Temporary notebook: `predict()` vs P(specialty | consult sequence)

This is a scratch notebook, not part of the numbered series. It illustrates a mismatch in `SequenceToOutcomePredictor`.

Notebook **3c** describes a two-stage tree:

1. snapshot consult sequence $X$ → distribution over **final** sequences
2. each final sequence → distribution over **specialty**

At inference, a patient is still in the ED, so we only observe $X$ (consults so far). `predict(X)` is meant to return **P(specialty | consultation sequence = $X$)**. This notebook uses a small constructed dataset to show that it may not.

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from datetime import date

from patientflow.predictors.sequence_to_outcome_predictor import SequenceToOutcomePredictor

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

## Demo data

Seven admitted patients, one snapshot each. Every `consultation_sequence` is a prefix of that patient's `final_sequence`, as in real ED data.

- **Alice, Bob, Carol** have no consults yet at the snapshot. Some later get consults; Alice never does.
- **Dave, Eve, Frank** already have an acute consult.
- **Grace** already has `(acute, icu)`, a sequence that never appears as anyone's *complete* final sequence.

In [2]:
demo = pd.DataFrame(
    [
        {"patient": "Alice", "consultation_sequence": (), "final_sequence": (), "specialty": "medical"},
        {"patient": "Bob", "consultation_sequence": (), "final_sequence": ("acute",), "specialty": "medical"},
        {"patient": "Carol", "consultation_sequence": (), "final_sequence": ("surgical",), "specialty": "surgical"},
        {"patient": "Dave", "consultation_sequence": ("acute",), "final_sequence": ("acute",), "specialty": "medical"},
        {"patient": "Eve", "consultation_sequence": ("acute",), "final_sequence": ("acute", "surgical"), "specialty": "surgical"},
        {"patient": "Frank", "consultation_sequence": ("acute",), "final_sequence": ("acute", "surgical"), "specialty": "surgical"},
        {"patient": "Grace", "consultation_sequence": ("acute", "icu"), "final_sequence": ("acute", "icu", "medical"), "specialty": "medical"},
    ]
)
demo["is_admitted"] = True
demo["snapshot_date"] = date(2024, 1, 1)
demo

,patient,consultation_sequence,final_sequence,specialty,is_admitted,snapshot_date
0,Alice,(),(),medical,True,2024-01-01
1,Bob,(),"(acute,)",medical,True,2024-01-01
2,Carol,(),"(surgical,)",surgical,True,2024-01-01
3,Dave,"(acute,)","(acute,)",medical,True,2024-01-01
4,Eve,"(acute,)","(acute, surgical)",surgical,True,2024-01-01
5,Frank,"(acute,)","(acute, surgical)",surgical,True,2024-01-01
6,Grace,"(acute, icu)","(acute, icu, medical)",medical,True,2024-01-01


## Fit the model

`weights` is what `predict()` looks up. If the model were doing what notebook 3c describes, those keys would be unique **consultation** sequences. They are unique **final** sequences instead.

In [3]:
model = SequenceToOutcomePredictor(
    input_var="consultation_sequence",
    grouping_var="final_sequence",
    outcome_var="specialty",
    apply_special_category_filtering=False,
)
model.fit(demo)

keys = pd.DataFrame(
    {
        "unique consultation_sequence": pd.Series(
            sorted(demo.consultation_sequence.unique(), key=lambda x: (len(x), x))
        ),
        "unique final_sequence": pd.Series(
            sorted(demo.final_sequence.unique(), key=lambda x: (len(x), x))
        ),
        "weights keys (used by predict)": pd.Series(
            sorted(model.weights.keys(), key=lambda x: (len(x), x))
        ),
    }
)
keys

,unique consultation_sequence,unique final_sequence,weights keys (used by predict)
0,(),(),()
1,"(acute,)","(acute,)","(acute,)"
2,"(acute, icu)","(surgical,)","(surgical,)"
3,NaN,"(acute, surgical)","(acute, surgical)"
4,NaN,"(acute, icu, medical)","(acute, icu, medical)"


The consult → final mapping *is* stored, as `input_to_grouping_probs`. Each row is P(final sequence | consult sequence). `predict()` does not use this table.

In [4]:
model.input_to_grouping_probs

final_sequence,(),"(acute,)","(acute, icu, medical)","(acute, surgical)","(surgical,)",probability_of_grouping_sequence
consultation_sequence,,,,,,
(),0.333333,0.333333,0.0,0.000000,0.333333,0.428571
"(acute,)",0.000000,0.333333,0.0,0.666667,0.000000,0.428571
"(acute, icu)",0.000000,0.000000,1.0,0.000000,0.000000,0.142857


## What `predict(X)` should return vs what it returns

For each snapshot sequence $X$:

- **Empirical** is the specialty mix among patients whose consults so far are exactly $X$. That is the target: P(specialty | $X$).
- **Tree** is the notebook 3c composition: $\sum_F$ P($F$ | $X$) P(specialty | $F$). On this dataset it matches the empirical values.
- **`predict(X)`** is the model output.

In [5]:
SPECIALTIES = ["medical", "surgical"]


def empirical_specialty_given_consult(df, seq):
    sub = df[df.consultation_sequence == seq]
    if sub.empty:
        return pd.Series(0.0, index=SPECIALTIES)
    return sub.specialty.value_counts(normalize=True).reindex(SPECIALTIES, fill_value=0.0)


def tree_specialty_given_consult(df, seq):
    """Notebook 3c: sum over finals of P(F|X) * P(specialty|F)."""
    p_spec_given_F = (
        df.groupby("final_sequence")["specialty"]
        .value_counts(normalize=True)
        .unstack(fill_value=0)
        .reindex(columns=SPECIALTIES, fill_value=0.0)
    )
    sub = df[df.consultation_sequence == seq]
    if sub.empty:
        return pd.Series(0.0, index=SPECIALTIES)
    p_F = sub.final_sequence.value_counts(normalize=True)
    return p_spec_given_F.reindex(p_F.index).mul(p_F, axis=0).sum().reindex(SPECIALTIES, fill_value=0.0)


def predict_as_series(seq):
    pred = model.predict(seq) or {}
    return pd.Series({s: pred.get(s, 0.0) for s in SPECIALTIES})


queries = [(), ("acute",), ("acute", "icu")]
rows = []
for seq in queries:
    n = int((demo.consultation_sequence == seq).sum())
    emp = empirical_specialty_given_consult(demo, seq)
    tree = tree_specialty_given_consult(demo, seq)
    pred = predict_as_series(seq)
    rows.append(
        {
            "consultation_sequence": seq,
            "n": n,
            "in_weights": seq in model.weights,
            "empirical_medical": emp["medical"],
            "tree_medical": tree["medical"],
            "predict_medical": pred["medical"],
            "empirical_surgical": emp["surgical"],
            "tree_surgical": tree["surgical"],
            "predict_surgical": pred["surgical"],
        }
    )

comparison = pd.DataFrame(rows).round(3)
comparison

,consultation_sequence,n,in_weights,empirical_medical,tree_medical,predict_medical,empirical_surgical,tree_surgical,predict_surgical
0,(),3,True,0.667,0.667,0.571,0.333,0.333,0.429
1,"(acute,)",3,True,0.333,0.333,0.600,0.667,0.667,0.400
2,"(acute, icu)",1,False,1.000,1.000,0.600,0.000,0.000,0.400


If `predict()` were correct, the `empirical_*` / `tree_*` columns would match the `predict_*` columns. They do not.

## Three inference cases

### 1. Empty consults: `predict(())`

Alice, Bob and Carol have no consults yet. Two of the three are later admitted under medicine, so P(medical | no consults yet) = 2/3.

`weights[()]` is built by prefix-matching **every** final sequence (the empty prefix matches all of them). That is the **overall** specialty mix, 4/7 medical, not the mix among people still without consults.

In [6]:
print("Patients with consultation_sequence = ():")
display(demo.loc[demo.consultation_sequence == (), ["patient", "final_sequence", "specialty"]])

print("Empirical P(specialty | no consults yet):")
print(empirical_specialty_given_consult(demo, ()).round(3).to_dict())

print("\nOverall specialty mix (all seven patients):")
print(demo.specialty.value_counts(normalize=True).round(3).to_dict())

print("\nmodel.predict(()):")
print({k: round(v, 3) for k, v in model.predict(()).items()})

Patients with consultation_sequence = ():


,patient,final_sequence,specialty
0,Alice,(),medical
1,Bob,"(acute,)",medical
2,Carol,"(surgical,)",surgical


Empirical P(specialty | no consults yet):
{'medical': 0.667, 'surgical': 0.333}

Overall specialty mix (all seven patients):
{'medical': 0.571, 'surgical': 0.429}

model.predict(()):
{'medical': 0.571, 'surgical': 0.429}


### 2. Consult sequence that also exists as a final sequence: `predict(("acute",))`

Dave, Eve and Frank currently have `(acute,)`. Two of the three end in surgery, so P(surgical | currently acute) = 2/3.

`predict(("acute",))` instead uses every patient whose **finished** sequence *starts with* acute. That includes Bob (no consults yet, later acute) and Grace (already `(acute, icu)`), which pulls the surgical probability down to 0.4.

In [7]:
print("Patients with consultation_sequence = ('acute',):")
display(demo.loc[demo.consultation_sequence == ("acute",), ["patient", "final_sequence", "specialty"]])

starts_with_acute = demo.final_sequence.apply(lambda f: f[:1] == ("acute",))
print("Patients whose final_sequence starts with ('acute',) — this is what weights uses:")
display(demo.loc[starts_with_acute, ["patient", "consultation_sequence", "final_sequence", "specialty"]])

print("Empirical P(specialty | currently ('acute',)):")
print(empirical_specialty_given_consult(demo, ("acute",)).round(3).to_dict())

print("\nP(specialty | final starts with ('acute',)):")
print(demo.loc[starts_with_acute, "specialty"].value_counts(normalize=True).round(3).to_dict())

print("\nmodel.predict(('acute',)):")
print({k: round(v, 3) for k, v in model.predict(("acute",)).items()})

Patients with consultation_sequence = ('acute',):


,patient,final_sequence,specialty
3,Dave,"(acute,)",medical
4,Eve,"(acute, surgical)",surgical
5,Frank,"(acute, surgical)",surgical


Patients whose final_sequence starts with ('acute',) — this is what weights uses:


,patient,consultation_sequence,final_sequence,specialty
1,Bob,(),"(acute,)",medical
3,Dave,"(acute,)","(acute,)",medical
4,Eve,"(acute,)","(acute, surgical)",surgical
5,Frank,"(acute,)","(acute, surgical)",surgical
6,Grace,"(acute, icu)","(acute, icu, medical)",medical


Empirical P(specialty | currently ('acute',)):
{'medical': 0.333, 'surgical': 0.667}

P(specialty | final starts with ('acute',)):
{'medical': 0.6, 'surgical': 0.4}

model.predict(('acute',)):
{'medical': 0.6, 'surgical': 0.4}


### 3. Snapshot sequence that never occurs as a complete final sequence: `predict(("acute", "icu"))`

Grace is the only patient at `(acute, icu)`. She is admitted under medicine, so the correct probability is medical = 1.

`(acute, icu)` is not a key in `weights` (it never appears as a `final_sequence`). `predict()` strips the last element and returns `predict(("acute",))` instead.

In [8]:
seq = ("acute", "icu")
print(f"{seq!r} in weights? {seq in model.weights}")
print(f"('acute',) in weights? {('acute',) in model.weights}")

print("\nEmpirical P(specialty | currently ('acute', 'icu')):")
print(empirical_specialty_given_consult(demo, seq).round(3).to_dict())

print("\nmodel.predict(('acute', 'icu'))  — falls back to ('acute',):")
print({k: round(v, 3) for k, v in model.predict(seq).items()})
print("model.predict(('acute',)):")
print({k: round(v, 3) for k, v in model.predict(("acute",)).items()})

('acute', 'icu') in weights? False
('acute',) in weights? True

Empirical P(specialty | currently ('acute', 'icu')):
{'medical': 1.0, 'surgical': 0.0}

model.predict(('acute', 'icu'))  — falls back to ('acute',):
{'medical': 0.6, 'surgical': 0.4}
model.predict(('acute',)):
{'medical': 0.6, 'surgical': 0.4}


## Conclusion

On this dataset, a patient part-way through an ED visit with consult sequence $X$ does **not** get P(specialty | $X$) from `predict(X)`.

| Snapshot $X$ | True P(medical $\mid$ $X$) | `predict(X)` medical |
|---|---|---|
| `()` | 0.667 | 0.571 |
| `('acute',)` | 0.333 | 0.600 |
| `('acute', 'icu')` | 1.000 | 0.600 |

What the model actually returns is P(specialty | **final sequence starts with** this key), keyed by completed sequences, with truncation if the snapshot sequence was never seen as a complete path.

`input_to_grouping_probs` already holds P(final | $X$). Composing that with P(specialty | final) — as `ValueToOutcomePredictor` does for scalar inputs — would recover the empirical values above.